# Inland Excess Water (IEW) Monitoring — Stage F: Temporal Gap-Filling

Implements **§3.2.2, Algorithm 2** of the paper: resolving cloud-obscured Sentinel-2 pixels by combining the binary **Radar_Model** with the nearest cloud-free S2 predictions **before** and **after** the gap.

```
Inputs: s2_prev (last cloud-free S2 result), s2_next (next cloud-free S2 result),
        s2_day (today's S2 result, has cloud gaps), radar_mask (today's binary radar result)

mask = (s2_day == missing)
s2_day[mask & (radar_mask == water)] = "water"        # radar fills in open-water pixels

if s2_prev == s2_next:
    s2_day = s2_prev                                    # stable -> use it directly
elif s2_prev or s2_next is missing:
    s2_day = whichever scan is available
else:
    s2_day = (s2_prev, s2_next)                          # disagreement -> SplitClass-style dual label
```

## ⚠️ Important honesty note before you read the numbers below

The paper's actual **Table 3** validates gap-filling against an independent **2024 field survey** (QField GPS points; BNPI 321 / KNPI 439 usable points after filtering). **That field-survey file is not present in this repository's dataset copy** (flagged already in `docs/PHASE_1_PRD.md` §2 and `docs/PAPER_WORKFLOW.md` §7) — so this notebook **cannot reproduce Table 3's exact numbers**, and does not claim to.

What it does instead, so the algorithm is genuinely implemented and evaluated rather than left as an unused function: it **synthetically treats whole test patches as fully cloud-obscured**, reconstructs them via Algorithm 2 using the chronologically nearest available scenes of the same site (BNPI/KNPI have 11-12 dates spanning 2021-2024, confirmed to share an identical raster grid per site — see below), and scores the reconstruction against the *real* ground truth for that patch (which we have, since it's the labeled test set). This is a **proxy evaluation of the algorithm's mechanics**, not a reproduction of the paper's field-validated accuracy — report it as such.

**Pipeline position:** `A → B (S2_Model + Radar_Model) → C → D → E → F. Temporal Gap-Filling (this notebook, final stage)`

**Prerequisites:** run `rupnet.ipynb` (or `04_knowledge_distillation.ipynb` for the paper-faithful student) and `06_radar_model.ipynb` first.

Flow: imports → data pipeline → load S2_Model + Radar_Model → build per-site chronological scene index → Algorithm 2 implementation → patch-level windowed prediction helpers → synthetic-gap evaluation (Single-Day vs. Gap-Filling vs. Oracle) → visualization.

## 1. Imports and Setup

In [ ]:
import os
import sys
import glob
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.windows import Window
from pathlib import Path
from typing import List, Dict, Tuple

torch.manual_seed(0)
np.random.seed(0)

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


## 2. Data Pipeline

In [ ]:
# --- Raster I/O helpers -----------------------------------------------------
def read_s2(path: str) -> np.ndarray:
    """Read a Sentinel-2 GeoTIFF -> (11, H, W) float32, scaled to reflectance [0, ~1.7]."""
    with rasterio.open(path) as src:
        data = src.read().astype(np.float32)
        data /= 10000.0  # S2 BOA reflectance is stored scaled by 10000
        return data

def read_s1(path: str) -> np.ndarray:
    """Read a Sentinel-1 GeoTIFF -> (3, H, W) float32: VV, VH, VV/VH ratio (dB)."""
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def read_gt(path: str) -> np.ndarray:
    """Read a ground-truth GeoTIFF -> (H, W) int class map."""
    with rasterio.open(path) as src:
        return src.read(1)


def get_scene_files(dataset_root: str, split: str) -> List[Dict[str, str]]:
    """Find and align S1, S2, GT, GT_0_8 files for a split ('Train'/'Test'), keyed off S2."""
    root = Path(dataset_root)
    split_dir = root / "tifs" / split

    s1_dir = split_dir / "S1"
    s2_dir = split_dir / "S2"
    gt_dir = split_dir / "GT"
    gt08_dir = split_dir / "GT_0_8"

    s2_files = sorted(split_dir.joinpath("S2").glob("*.tif"))

    aligned = []
    for s2_file in s2_files:
        name = s2_file.name
        s1_path = s1_dir / name
        gt_path = gt_dir / name
        gt08_path = gt08_dir / name

        if not s1_path.exists():
            raise FileNotFoundError(f"Missing S1 file: {s1_path}")
        if not gt_path.exists():
            raise FileNotFoundError(f"Missing GT file: {gt_path}")
        if not gt08_path.exists():
            raise FileNotFoundError(f"Missing GT_0_8 file: {gt08_path}")

        aligned.append({
            "name": s2_file.stem,
            "S1": str(s1_path),
            "S2": str(s2_file),
            "GT": str(gt_path),
            "GT_0_8": str(gt08_path),
        })
    return aligned

print("Raster I/O helpers ready: read_s2, read_s1, read_gt, get_scene_files")


In [ ]:
# --- PyTorch Dataset ---------------------------------------------------------
# Per-channel statistics computed over the training split.
S2_MEAN = np.array([0.0436, 0.0557, 0.0723, 0.0838, 0.1151, 0.1598, 0.1800, 0.1974, 0.2004, 0.2233, 0.1706], dtype=np.float32)
S2_STD  = np.array([0.0182, 0.0233, 0.0272, 0.0340, 0.0380, 0.0598, 0.0686, 0.0751, 0.0728, 0.0674, 0.0606], dtype=np.float32)

S1_MEAN = np.array([-21.6081, -13.6686, -8.0403], dtype=np.float32)
S1_STD  = np.array([3.4706, 3.0644, 2.5309], dtype=np.float32)


class FloodPatchDataset(Dataset):
    """Sliding-window dataset over aligned S1/S2/GT scenes. Reads windows directly from
    disk via rasterio (or caches full scenes in memory), standardizes per-channel, and
    optionally applies flip/rotation augmentation.
    """

    def __init__(
        self,
        dataset_root: str,
        split: str,
        patch_size: int = 64,
        stride: int = 64,
        augment: bool = False,
        cache_in_memory: bool = False,
        load_s1: bool = True,
    ):
        self.dataset_root = dataset_root
        self.split = split
        self.patch_size = patch_size
        self.stride = stride
        self.augment = augment
        self.cache_in_memory = cache_in_memory
        self.load_s1 = load_s1

        self.scenes = get_scene_files(dataset_root, split)

        self.s2_cache, self.s1_cache, self.gt_cache = {}, {}, {}
        self.patches = []

        print(f"Initializing {split} dataset (cache_in_memory={cache_in_memory}, load_s1={load_s1})...")

        for scene_idx, scene in enumerate(self.scenes):
            with rasterio.open(scene["S2"]) as s2_src:
                H, W = s2_src.height, s2_src.width

            if cache_in_memory:
                with rasterio.open(scene["S2"]) as s2_src:
                    s2_img = s2_src.read().astype(np.float32)
                    s2_img /= 10000.0
                    self.s2_cache[scene_idx] = s2_img
                if load_s1:
                    with rasterio.open(scene["S1"]) as s1_src:
                        self.s1_cache[scene_idx] = s1_src.read().astype(np.float32)
                with rasterio.open(scene["GT_0_8"]) as gt_src:
                    self.gt_cache[scene_idx] = gt_src.read(1).astype(np.int64)
            # else: no persistent file handles are opened here - see __getitem__, which
            # opens+reads+closes per access. This keeps the Dataset free of unpicklable
            # state (required for num_workers > 0, since DataLoader workers get a pickled
            # copy of the Dataset) and avoids GDAL's internal per-handle cache growing
            # unbounded over a long-running, many-epoch loop that reuses the same handle.

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    self.patches.append({"scene_idx": scene_idx, "y": y, "x": x})

        print(f"Loaded {split} dataset with {len(self.patches)} total patches from {len(self.scenes)} scenes.")

    def __len__(self) -> int:
        return len(self.patches)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        info = self.patches[idx]
        scene_idx, y, x = info["scene_idx"], info["y"], info["x"]

        if self.cache_in_memory:
            s2_data = self.s2_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            if self.load_s1:
                s1_data = self.s1_cache[scene_idx][:, y:y + self.patch_size, x:x + self.patch_size].copy()
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            gt_data = self.gt_cache[scene_idx][y:y + self.patch_size, x:x + self.patch_size].copy()
        else:
            scene = self.scenes[scene_idx]
            window = Window(x, y, self.patch_size, self.patch_size)
            with rasterio.open(scene["S2"]) as s2_src:
                s2_data = s2_src.read(window=window).astype(np.float32)
            s2_data /= 10000.0
            if self.load_s1:
                with rasterio.open(scene["S1"]) as s1_src:
                    s1_data = s1_src.read(window=window).astype(np.float32)
            else:
                s1_data = np.zeros((3, self.patch_size, self.patch_size), dtype=np.float32)
            with rasterio.open(scene["GT_0_8"]) as gt_src:
                gt_data = gt_src.read(1, window=window).astype(np.int64)

        for c in range(11):
            s2_data[c] = (s2_data[c] - S2_MEAN[c]) / S2_STD[c]
        if self.load_s1:
            for c in range(3):
                s1_data[c] = (s1_data[c] - S1_MEAN[c]) / S1_STD[c]

        if self.augment:
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=2).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=2).copy()
                gt_data = np.flip(gt_data, axis=1).copy()
            if np.random.rand() > 0.5:
                s2_data = np.flip(s2_data, axis=1).copy()
                if self.load_s1:
                    s1_data = np.flip(s1_data, axis=1).copy()
                gt_data = np.flip(gt_data, axis=0).copy()
            rot_k = np.random.randint(0, 4)
            if rot_k > 0:
                s2_data = np.rot90(s2_data, k=rot_k, axes=(1, 2)).copy()
                if self.load_s1:
                    s1_data = np.rot90(s1_data, k=rot_k, axes=(1, 2)).copy()
                gt_data = np.rot90(gt_data, k=rot_k, axes=(0, 1)).copy()

        return torch.from_numpy(s2_data), torch.from_numpy(s1_data), torch.from_numpy(gt_data)


print("FloodPatchDataset ready")


In [ ]:
def stretch_rgb(rgb: np.ndarray, low: float = 2.0, high: float = 98.0) -> np.ndarray:
    """Per-channel percentile contrast stretch for a true-color composite (see rupnet.ipynb
    diagnosis note - a fixed divisor washes scenes out or leaves them too dark)."""
    out = np.empty_like(rgb, dtype=np.float32)
    for c in range(rgb.shape[-1]):
        lo, hi = np.percentile(rgb[..., c], [low, high])
        out[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)
    return out


print("stretch_rgb ready")


## 3. Model Architecture — RUPNet (shared by S2_Model and Radar_Model)

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, padding=1, stride=1, dilation=1, bias=False, act=True):
        super().__init__()
        self.act = act
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size, stride=stride, padding=padding, dilation=dilation, bias=bias),
            nn.BatchNorm2d(out_c),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        if self.act:
            x = self.relu(x)
        return x


class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
        )
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=1, bias=False),
                nn.BatchNorm2d(out_c),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))


class DecoderBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c + out_c, out_c)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x)


class UpBlock(nn.Module):
    def __init__(self, in_c, out_c, scale):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale, mode="bilinear", align_corners=True)
        self.res = ResidualBlock(in_c, out_c)

    def forward(self, x):
        return self.res(self.up(x))


class RUPNet(nn.Module):
    """RUPNet (Tomar, Bagci & Jha, 2023) - the paper's winning backbone (Table 1,
    F1 = 81.19%) and its knowledge-distillation teacher/student architecture (~2.05M params
    at in_channels=11; scales down automatically for fewer input channels, e.g. the 3-channel
    Radar_Model)."""

    def __init__(self, in_channels: int = 11, num_classes: int = 6):
        super().__init__()

        self.init_conv = Conv2D(in_channels, 32, kernel_size=3, padding=1)

        self.enc1_1 = ResidualBlock(32, 64)
        self.down1 = nn.MaxPool2d(2)
        self.enc1_2 = ResidualBlock(64, 64)
        self.down2 = nn.MaxPool2d(2)  # scale: H/4

        self.enc2 = ResidualBlock(64, 128)
        self.down3 = nn.MaxPool2d(2)  # scale: H/8

        self.enc3 = ResidualBlock(128, 256)
        self.down4 = nn.MaxPool2d(2)  # scale: H/16

        self.c1 = Conv2D(64, 64, kernel_size=1, padding=0)
        self.c2 = Conv2D(128, 64, kernel_size=1, padding=0)
        self.c3 = Conv2D(256, 64, kernel_size=1, padding=0)

        self.d1 = DecoderBlock(64, 64)
        self.d2 = DecoderBlock(64, 64)

        self.d3 = UpBlock(64, 64, scale=4)
        self.u1 = UpBlock(64, 64, scale=4)
        self.u2 = UpBlock(64, 64, scale=8)
        self.u3 = UpBlock(64, 64, scale=16)

        self.fusion = ResidualBlock(256, 64)
        self.outc = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, inputs):
        x = self.init_conv(inputs)

        e1 = self.enc1_1(x)
        e1 = self.down1(e1)
        e1 = self.enc1_2(e1)
        e1 = self.down2(e1)  # (B, 64, H/4, W/4)

        e2 = self.enc2(e1)
        e2 = self.down3(e2)  # (B, 128, H/8, W/8)

        e3 = self.enc3(e2)
        e3 = self.down4(e3)  # (B, 256, H/16, W/16)

        c1 = self.c1(e1)
        c2 = self.c2(e2)
        c3 = self.c3(e3)

        d1 = self.d1(c3, c2)  # (B, 64, H/8, W/8)
        d2 = self.d2(d1, c1)  # (B, 64, H/4, W/4)

        d3 = self.d3(d2)  # (B, 64, H, W)
        u1 = self.u1(c1)  # (B, 64, H, W)
        u2 = self.u2(c2)  # (B, 64, H, W)
        u3 = self.u3(c3)  # (B, 64, H, W)

        fused = torch.cat([d3, u1, u2, u3], dim=1)  # (B, 256, H, W)
        fused = self.fusion(fused)  # (B, 64, H, W)

        return self.outc(fused)


print("RUPNet ready")


## 4. Load S2_Model and Radar_Model

In [ ]:
with open("configs/data.yaml", "r") as f:
    config = yaml.safe_load(f)
dataset_root = config["dataset_root"]
PATCH_SIZE = config["patch_size"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

STUDENT_CKPT = "outputs/checkpoints/rupnet_student_best.pt"
TEACHER_CKPT = "outputs/checkpoints/rupnet_best.pt"
RADAR_CKPT = "outputs/checkpoints/radar_model_best.pt"

if os.path.exists(STUDENT_CKPT):
    s2_ckpt_path, S2_SOURCE = STUDENT_CKPT, "Stage-D Student"
elif os.path.exists(TEACHER_CKPT):
    s2_ckpt_path, S2_SOURCE = TEACHER_CKPT, "Stage-C Teacher-only (fallback)"
else:
    raise FileNotFoundError("Run rupnet.ipynb (and ideally 04_knowledge_distillation.ipynb) first.")

if not os.path.exists(RADAR_CKPT):
    raise FileNotFoundError("outputs/checkpoints/radar_model_best.pt not found - run 06_radar_model.ipynb first.")

s2_model = RUPNet(in_channels=11, num_classes=6).to(device)
s2_ckpt = torch.load(s2_ckpt_path, map_location=device)
s2_model.load_state_dict(s2_ckpt["model_state_dict"])
s2_model.eval()
print(f"S2_Model: loaded {S2_SOURCE} from {s2_ckpt_path}")

radar_model = RUPNet(in_channels=3, num_classes=2).to(device)
radar_ckpt = torch.load(RADAR_CKPT, map_location=device)
radar_model.load_state_dict(radar_ckpt["model_state_dict"])
radar_model.eval()
print(f"Radar_Model: loaded from {RADAR_CKPT}")

class_names = [
    "Open water (2)", "Soil seriously affected (3)", "Soil moderately affected (4)",
    "Soil slightly affected (5)", "Vegetation affected (6)", "Not affected (7)",
]


## 5. Per-Site Chronological Scene Index
BNPI and KNPI scenes share one fixed raster grid per site across all dates (verified: every BNPI scene is 5400x3600, every KNPI scene is 4800x2400) — so a patch window `(y, x)` refers to the *same ground location* regardless of which date's scene it's read from. This is what makes cross-date "prev/next" patch lookups valid.

In [ ]:
def parse_date(scene_name: str) -> int:
    return int(scene_name.split("_")[1])

def site_of(scene_name: str) -> str:
    return scene_name.split("_")[0]

all_scenes = get_scene_files(dataset_root, "Train") + get_scene_files(dataset_root, "Test")
test_scenes = get_scene_files(dataset_root, "Test")

site_chron = {}
for scene in all_scenes:
    site_chron.setdefault(site_of(scene["name"]), []).append(scene)
for site in site_chron:
    site_chron[site].sort(key=lambda s: parse_date(s["name"]))

for site, scenes in site_chron.items():
    print(f"{site}: {len(scenes)} dates -> {[s['name'] for s in scenes]}")


## 6. Patch-Level Windowed Prediction Helpers
Read a single patch window directly from any scene's GeoTIFF (not via the training `Dataset`, since we need to reach across different dates' files) and run it through a model.

In [ ]:
def predict_s2_patch(model, scene, y, x, size):
    """Returns the S2_Model's top-1 class map (size, size) int64 for this scene/window,
    or None if scene is None (i.e. no such date available)."""
    if scene is None:
        return None
    with rasterio.open(scene["S2"]) as src:
        data = src.read(window=Window(x, y, size, size)).astype(np.float32) / 10000.0
    for c in range(11):
        data[c] = (data[c] - S2_MEAN[c]) / S2_STD[c]
    t = torch.from_numpy(data).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(t), dim=1).squeeze(0).cpu().numpy()
    return pred


def predict_radar_patch(model, scene, y, x, size):
    """Returns the Radar_Model's binary prediction (size, size) int64: 1=water-affected, 0=not."""
    with rasterio.open(scene["S1"]) as src:
        data = src.read(window=Window(x, y, size, size)).astype(np.float32)
    for c in range(3):
        data[c] = (data[c] - S1_MEAN[c]) / S1_STD[c]
    t = torch.from_numpy(data).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.argmax(model(t), dim=1).squeeze(0).cpu().numpy()
    return pred


def read_gt08_window(scene, y, x, size):
    with rasterio.open(scene["GT_0_8"]) as src:
        return src.read(1, window=Window(x, y, size, size)).astype(np.int64)


def read_s2_rgb_window(scene, y, x, size):
    with rasterio.open(scene["S2"]) as src:
        data = src.read(window=Window(x, y, size, size)).astype(np.float32) / 10000.0
    return stretch_rgb(np.stack([data[3], data[2], data[1]], axis=-1))


print("Patch-level prediction helpers ready")


## 7. Algorithm 2 — Temporal Gap-Filling

In [ ]:
def temporal_gap_fill(prev_pred, next_pred, radar_pred):
    """Algorithm 2, applied to a fully cloud-obscured patch (this notebook's synthetic-gap
    evaluation scenario - see the honesty note in §0). Returns (top1, top2, is_dual):
    the same SplitClass-style composite-label representation used in 05_splitclass_segmentation.ipynb,
    since Algorithm 2's "keep both classes on disagreement" step is explicitly the same idea.
    """
    shape = radar_pred.shape
    # Step 1: radar-fill base layer (water where radar says water, "not affected" elsewhere) -
    # this is the ONLY information available if neither prev nor next exists.
    top1 = np.where(radar_pred == 1, 0, 5).astype(np.int64)   # 0 = open water, 5 = not affected
    top2 = top1.copy()
    is_dual = np.zeros(shape, dtype=bool)

    # Step 2: prev/next branch supersedes the radar fill wherever temporal context exists
    # (mirrors the paper's pseudocode, which unconditionally reassigns s2_day in this branch).
    if prev_pred is not None and next_pred is not None:
        agree = prev_pred == next_pred
        top1 = np.where(agree, prev_pred, prev_pred)   # top1 is prev's class either way
        top2 = np.where(agree, prev_pred, next_pred)   # top2 == top1 when they agree (no-op)
        is_dual = ~agree
    elif prev_pred is not None:
        top1, top2, is_dual = prev_pred, prev_pred, np.zeros(shape, dtype=bool)
    elif next_pred is not None:
        top1, top2, is_dual = next_pred, next_pred, np.zeros(shape, dtype=bool)
    # else: keep the Step-1 radar-only fallback

    return top1, top2, is_dual


print("Algorithm 2 (temporal_gap_fill) ready")


## 8. Synthetic-Gap Evaluation — Single-Day vs. Gap-Filling vs. Oracle
For each sampled non-overlapping patch in each test scene, pretend the *entire* patch is cloud-obscured on that date and compare three reconstructions against the real ground truth:

- **Single-Day** — the paper's literal fallback under cloud: binary Radar_Model prediction only (water → class "open water", else "not affected"). This is the number Gap-Filling is meant to improve on.
- **Gap-Filling** — Algorithm 2 (this notebook), using the same radar prediction plus the chronologically nearest prev/next S2 predictions.
- **Oracle** — the S2_Model's *actual* prediction on the real (non-obscured) patch for that date. Not a realistic fallback (if it were really cloudy you would not have this), included only as an upper-bound reference so the gap you're closing is visible.

Scoring uses the same "either assigned class matches GT" rule as SplitClass for dual-labeled Gap-Filling pixels.

In [ ]:
N_SAMPLES_PER_SCENE = 40
rng = np.random.default_rng(0)

per_method_correct = {"single_day": 0, "gap_filled": 0, "oracle": 0}
per_method_total = {"single_day": 0, "gap_filled": 0, "oracle": 0}
per_site = {}  # site -> per_method dicts, same structure
regime_pixel_counts = {"prev_next_agree": 0, "one_side_only": 0, "disagree_dual": 0, "both_missing": 0}

for test_scene in test_scenes:
    site = site_of(test_scene["name"])
    chron = site_chron[site]
    idx = [s["name"] for s in chron].index(test_scene["name"])
    prev_scene = chron[idx - 1] if idx > 0 else None
    next_scene = chron[idx + 1] if idx < len(chron) - 1 else None

    per_site.setdefault(site, {"single_day": [0, 0], "gap_filled": [0, 0], "oracle": [0, 0]})

    with rasterio.open(test_scene["S2"]) as src:
        H, W = src.height, src.width
    ys = list(range(0, H - PATCH_SIZE + 1, PATCH_SIZE))
    xs = list(range(0, W - PATCH_SIZE + 1, PATCH_SIZE))
    grid = [(y, x) for y in ys for x in xs]
    sample_idx = rng.choice(len(grid), size=min(N_SAMPLES_PER_SCENE, len(grid)), replace=False)

    for gi in sample_idx:
        y, x = grid[gi]
        gt_patch = read_gt08_window(test_scene, y, x, PATCH_SIZE)
        valid = gt_patch < 6
        if valid.sum() == 0:
            continue

        radar_pred = predict_radar_patch(radar_model, test_scene, y, x, PATCH_SIZE)
        prev_pred = predict_s2_patch(s2_model, prev_scene, y, x, PATCH_SIZE)
        next_pred = predict_s2_patch(s2_model, next_scene, y, x, PATCH_SIZE)
        oracle_pred = predict_s2_patch(s2_model, test_scene, y, x, PATCH_SIZE)  # real, uncloud "today"

        top1, top2, is_dual = temporal_gap_fill(prev_pred, next_pred, radar_pred)

        single_pred = np.where(radar_pred == 1, 0, 5).astype(np.int64)
        single_correct = np.sum((single_pred == gt_patch) & valid)
        gap_correct = np.sum(((top1 == gt_patch) | (is_dual & (top2 == gt_patch))) & valid)
        oracle_correct = np.sum((oracle_pred == gt_patch) & valid)
        n_valid = int(valid.sum())

        for key, correct in [("single_day", single_correct), ("gap_filled", gap_correct), ("oracle", oracle_correct)]:
            per_method_correct[key] += int(correct)
            per_method_total[key] += n_valid
            per_site[site][key][0] += int(correct)
            per_site[site][key][1] += n_valid

        if prev_pred is not None and next_pred is not None:
            regime_pixel_counts["prev_next_agree"] += int((prev_pred == next_pred).sum())
            regime_pixel_counts["disagree_dual"] += int((prev_pred != next_pred).sum())
        elif prev_pred is not None or next_pred is not None:
            regime_pixel_counts["one_side_only"] += gt_patch.size
        else:
            regime_pixel_counts["both_missing"] += gt_patch.size

print(f"Evaluated {per_method_total['gap_filled']:,} valid pixels "
      f"across {len(test_scenes)} test scenes ({N_SAMPLES_PER_SCENE} sampled patches/scene).\n")

print("=" * 60)
print(f"{'Method':<20}{'Accuracy (%)':<15}")
print("=" * 60)
for key, label in [("single_day", "Single-Day (radar only)"), ("gap_filled", "Gap-Filling (Alg. 2)"), ("oracle", "Oracle (uncloud today)")]:
    acc = 100 * per_method_correct[key] / max(per_method_total[key], 1)
    print(f"{label:<20}{acc:<15.2f}")
print("=" * 60)
print("\nBy site:")
print(f"{'Site':<8}{'Single-Day':<14}{'Gap-Filling':<14}{'Oracle':<10}")
for site, m in per_site.items():
    row = "".join(f"{100*m[k][0]/max(m[k][1],1):<14.2f}" if k != "oracle" else f"{100*m[k][0]/max(m[k][1],1):<10.2f}" for k in ["single_day", "gap_filled", "oracle"])
    print(f"{site:<8}{row}")

print("\nPaper reference (Table 3, real field-validated data): "
      "Single-Day BNPI=58.25% KNPI=59.45%  |  Gap-Filling BNPI=72.27% KNPI=69.24%")
print("(Not directly comparable to the synthetic-gap numbers above - see the \xa70 honesty note.)")

total_regime = sum(regime_pixel_counts.values())
print("\nTemporal-context regime breakdown (pixel-level, across all sampled patches):")
for k, v in regime_pixel_counts.items():
    print(f"  {k:<20} {100*v/max(total_regime,1):5.2f}%  ({v:,} px)")


## 9. Visualization — One Worked Example

In [ ]:
# Find a test scene/patch where prev and next actually disagree, so the dual-label
# mechanism is visible in the figure (falls back to the first available patch otherwise).
example = None
for test_scene in test_scenes:
    site = site_of(test_scene["name"])
    chron = site_chron[site]
    idx = [s["name"] for s in chron].index(test_scene["name"])
    prev_scene = chron[idx - 1] if idx > 0 else None
    next_scene = chron[idx + 1] if idx < len(chron) - 1 else None
    if prev_scene is None or next_scene is None:
        continue
    with rasterio.open(test_scene["S2"]) as src:
        H, W = src.height, src.width
    for y in range(0, H - PATCH_SIZE + 1, PATCH_SIZE * 4):
        for x in range(0, W - PATCH_SIZE + 1, PATCH_SIZE * 4):
            gt_patch = read_gt08_window(test_scene, y, x, PATCH_SIZE)
            if (gt_patch < 6).sum() == 0:
                continue
            prev_pred = predict_s2_patch(s2_model, prev_scene, y, x, PATCH_SIZE)
            next_pred = predict_s2_patch(s2_model, next_scene, y, x, PATCH_SIZE)
            if (prev_pred != next_pred).mean() > 0.15:  # meaningfully disagreeing patch
                example = (test_scene, prev_scene, next_scene, y, x, gt_patch, prev_pred, next_pred)
                break
        if example:
            break
    if example:
        break

if example is None:
    print("No strongly-disagreeing example patch found in this run - skipping visualization.")
else:
    test_scene, prev_scene, next_scene, y, x, gt_patch, prev_pred, next_pred = example
    radar_pred = predict_radar_patch(radar_model, test_scene, y, x, PATCH_SIZE)
    oracle_pred = predict_s2_patch(s2_model, test_scene, y, x, PATCH_SIZE)
    top1, top2, is_dual = temporal_gap_fill(prev_pred, next_pred, radar_pred)
    single_pred = np.where(radar_pred == 1, 0, 5).astype(np.int64)
    rgb_today = read_s2_rgb_window(test_scene, y, x, PATCH_SIZE)

    print(f"Example: today={test_scene['name']}  prev={prev_scene['name']}  next={next_scene['name']}  window=({y},{x})")

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes[0, 0].imshow(rgb_today); axes[0, 0].set_title(f"Today RGB ({test_scene['name']})"); axes[0, 0].axis("off")
    axes[0, 1].imshow(gt_patch, cmap="tab10", vmin=0, vmax=5); axes[0, 1].set_title("Ground Truth"); axes[0, 1].axis("off")
    axes[0, 2].imshow(single_pred, cmap="tab10", vmin=0, vmax=5); axes[0, 2].set_title("Single-Day (radar only)"); axes[0, 2].axis("off")
    axes[0, 3].imshow(top1, cmap="tab10", vmin=0, vmax=5); axes[0, 3].set_title("Gap-Filling: Top-1"); axes[0, 3].axis("off")

    axes[1, 0].imshow(prev_pred, cmap="tab10", vmin=0, vmax=5); axes[1, 0].set_title(f"S2 pred: prev ({prev_scene['name']})"); axes[1, 0].axis("off")
    axes[1, 1].imshow(next_pred, cmap="tab10", vmin=0, vmax=5); axes[1, 1].set_title(f"S2 pred: next ({next_scene['name']})"); axes[1, 1].axis("off")
    dual_overlay = np.ma.masked_where(~is_dual, is_dual.astype(float))
    axes[1, 2].imshow(top1, cmap="tab10", vmin=0, vmax=5, alpha=0.5)
    axes[1, 2].imshow(dual_overlay, cmap="gray_r", vmin=0, vmax=1, alpha=0.9)
    axes[1, 2].set_title("Dual-labeled pixels (prev != next)"); axes[1, 2].axis("off")
    axes[1, 3].imshow(oracle_pred, cmap="tab10", vmin=0, vmax=5); axes[1, 3].set_title("Oracle (uncloud today)"); axes[1, 3].axis("off")

    plt.tight_layout()
    plt.show()


## 10. Summary — fill in after running

- **Single-Day accuracy (synthetic-gap proxy):** ___%  **Gap-Filling accuracy:** ___%  **Oracle:** ___%
- **Does Gap-Filling beat Single-Day, as the paper claims?** ___
- **% pixels where prev/next agreed (stable, single class used):** ___
- **% pixels dual-labeled (prev/next disagreed):** ___
- **Caveat to repeat in any report:** these numbers are a *synthetic whole-patch cloud simulation*, not the paper's field-survey-validated Table 3. To close that gap, the project would need the 2024 QField field-survey file (BNPI/KNPI GPS points + class + depth), which is not present in this repository's dataset copy — see `docs/PHASE_1_PRD.md` §2.
- **All four new-stage checkpoints now exist:** `rupnet_student_best.pt` (D), `radar_model_best.pt` (B/F) — SplitClass (E) is a post-processing step with no checkpoint of its own.